# Dense NN Low-Pass Direct Search

This notebook is the standalone model-first follow-up to `simulation_small_sclae.ipynb`.
It optimizes simulator settings for `dense_nn_gpu` under `lowpass = true` by training on simulated data and scoring mean `balanced_accuracy` on the real labeled dataset.

What changed relative to the old in-notebook direct-search section:
- the search no longer reuses top settings from the simulator-first workflow
- the baseline is always the normal `base_cfg`
- `broad_random`, `latin_hypercube`, and `monte_carlo` all search directly for better real-data Dense-NN performance

The search still spans the same 48-dimensional simulator space, including the exposed `tilted_bar_hanning_length` and `one_sided_fan_*` basis-shape parameters.


## Environment Setup

Run the next cell once after opening the notebook or restarting the Julia kernel.
It activates the project, loads the helper module, and prints a few sanity checks.


In [1]:
# Environment setup for the notebook.
import Pkg

ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

Pkg.activate(joinpath(pwd(), "..", "data_generation"))

using Flux
using MLUtils
using StatsBase
using CairoMakie
using Random
using Distributions
using Statistics
using DecisionTree
using LIBSVM
using CUDA
using cuDNN
using CSV
using DataFrames
using HDF5

if !isdefined(Main, :ERPGen)
    include(joinpath(pwd(), "..", "data_generation", "erpgen.jl"))
end
using .ERPGen

if !isdefined(Main, :ERPImageUtils)
    include(joinpath(pwd(), "..", "utils", "erp_image_utils.jl"))
end
using .ERPImageUtils

include(joinpath(pwd(), "simulation_small_scale_helpers.jl"))
using .SmallScaleERPClassification

println("Active project: ", Base.active_project())
println("Flux version: ", Base.pkgversion(Flux))
println("CUDA functional: ", CUDA.functional())


  Activating project at `~/Dokumente/BA2/notebooks/data_generation`


Active project: /home/benjamin/Dokumente/BA2/notebooks/data_generation/Project.toml
Flux version: 0.16.9
CUDA functional: true


## Search Controls

Edit the next cell before launching the heavy run.
The three search methods are evaluated against the same real-data objective: mean `balanced_accuracy` of the Dense NN on the labeled low-pass dataset.


In [2]:
# Dense-NN direct-search controls.
BASE_SEED = Int(time_ns())

NN_DIRECT_TARGET_SIZE = (16, 16)
NN_DIRECT_LOWPASS = true
NN_DIRECT_POST_STIM_ONLY = true
NN_DIRECT_N_PER_PATTERN = 1000
NN_DIRECT_EVAL_REPEATS = 3
NN_DIRECT_LOCAL_RANGE_SCALE = 0.25

NN_DIRECT_METHOD_CANDIDATES = Dict(
    :broad_random => 12,
    :latin_hypercube => 12,
    :monte_carlo => 12,
)

NN_BATCH_SIZE = 32
NN_EPOCHS = 30
NN_LR = 1f-3

println("Seed: ", BASE_SEED)
println("Target size: ", NN_DIRECT_TARGET_SIZE)
println("Low-pass: ", NN_DIRECT_LOWPASS)
println("Post-stim only: ", NN_DIRECT_POST_STIM_ONLY)
println("Simulated samples per class: ", NN_DIRECT_N_PER_PATTERN)
println("Eval repeats per candidate: ", NN_DIRECT_EVAL_REPEATS)
println("Local range scale: ", NN_DIRECT_LOCAL_RANGE_SCALE)
println("Method candidate budget: ", NN_DIRECT_METHOD_CANDIDATES)


Seed: 1191262759966
Target size: (16, 16)
Low-pass: true
Post-stim only: true
Simulated samples per class: 1000
Eval repeats per candidate: 3
Local range scale: 0.25
Method candidate budget: Dict(:monte_carlo => 12, :broad_random => 12, :latin_hypercube => 12)


## Run Direct Search

The next cell runs the standalone direct search.
It evaluates `base_cfg` once as the baseline and then scores fresh candidates from `broad_random`, `latin_hypercube`, and `monte_carlo` on the real labeled dataset.


In [3]:
# Run the standalone Dense-NN low-pass direct search.
dense_nn_direct_search = run_dense_nn_lowpass_direct_search(
    target_size = NN_DIRECT_TARGET_SIZE,
    lowpass = NN_DIRECT_LOWPASS,
    method_candidates = NN_DIRECT_METHOD_CANDIDATES,
    n_per_pattern = NN_DIRECT_N_PER_PATTERN,
    eval_repeats = NN_DIRECT_EVAL_REPEATS,
    local_range_scale = NN_DIRECT_LOCAL_RANGE_SCALE,
    nn_batch_size = NN_BATCH_SIZE,
    nn_epochs = NN_EPOCHS,
    nn_lr = NN_LR,
    post_stim_only = NN_DIRECT_POST_STIM_ONLY,
    seed = BASE_SEED,
)


  real images processed: 50/488
  real images processed: 100/488
  real images processed: 150/488
  real images processed: 200/488
  real images processed: 250/488
  real images processed: 300/488
  real images processed: 350/488
  real images processed: 400/488
  real images processed: 450/488
  real images processed: 488/488

Dense NN low-pass direct search | baseline=base_cfg
    dense_nn epoch 1/30 | loss=0.44619
    dense_nn epoch 6/30 | loss=0.03818
    dense_nn epoch 12/30 | loss=0.00983
    dense_nn epoch 18/30 | loss=0.00879
    dense_nn epoch 24/30 | loss=0.01711
    dense_nn epoch 30/30 | loss=0.01207
    dense_nn epoch 1/30 | loss=0.45032
    dense_nn epoch 6/30 | loss=0.03781
    dense_nn epoch 12/30 | loss=0.02434
    dense_nn epoch 18/30 | loss=0.00433
    dense_nn epoch 24/30 | loss=0.0451
    dense_nn epoch 30/30 | loss=0.03389
    dense_nn epoch 1/30 | loss=0.4406
    dense_nn epoch 6/30 | loss=0.03587
    dense_nn epoch 12/30 | loss=0.03609
    dense_nn epoch 18/30 |

(baseline_results_df = 1×74 DataFrame
 Row │ search_method  resolution  lowpass  candidate_index  candidate_source   ⋯
     │ String         String      Bool     Int64            String             ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ baseline       16x16          true                1  baseline           ⋯
                                                              69 columns omitted, results_df = 36×74 DataFrame
 Row │ search_method  resolution  lowpass  candidate_index  candidate_source   ⋯
     │ String         String      Bool     Int64            String             ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ broad_random   16x16          true                7  global_random      ⋯
   2 │ broad_random   16x16          true                6  global_random
   3 │ broad_random   16x16          true               11  global_random
   4 │ broad_random   16x16          true              

## Baseline And Leaderboards

Read the next cell in this order:
- `baseline_results_df`: how strong the default `base_cfg` already is
- `summary_df`: the best discovered candidate per method
- `results_df`: the ranked discovered candidates themselves


In [4]:
# Inspect the baseline and method-specific leaderboards.
display(dense_nn_direct_search.baseline_results_df[:, [
    :search_method,
    :candidate_source,
    :seed_label,
    :mean_balanced_accuracy,
    :mean_macro_f1,
    :mean_train_time_s,
]])

display(dense_nn_direct_search.summary_df)

first(
    dense_nn_direct_search.results_df[:, [
        :search_method,
        :method_rank,
        :candidate_source,
        :seed_label,
        :mean_balanced_accuracy,
        :std_balanced_accuracy,
        :mean_macro_f1,
        :mean_train_time_s,
    ]],
    18,
)


Row,search_method,candidate_source,seed_label,mean_balanced_accuracy,mean_macro_f1,mean_train_time_s
,String,String,String,Float64,Float64,Float64
1,baseline,baseline,base_cfg,0.673836,0.600926,13.169


Row,search_method,best_balanced_accuracy,best_macro_f1,n_evaluated
,String,Float64,Float64,Int64
1,latin_hypercube,0.710955,0.658715,12
2,monte_carlo,0.710505,0.636314,12
3,broad_random,0.691658,0.657282,12


Row,search_method,method_rank,candidate_source,seed_label,mean_balanced_accuracy,std_balanced_accuracy,mean_macro_f1,mean_train_time_s
,String,Int64?,String,String,Float64,Float64,Float64,Float64
1,broad_random,1,global_random,base_cfg,0.691658,0.0324754,0.657282,2.20594
2,broad_random,2,global_random,base_cfg,0.662611,0.0170341,0.538808,2.1744
3,broad_random,3,global_random,base_cfg,0.642576,0.0483045,0.538631,2.12949
4,broad_random,4,global_random,base_cfg,0.637619,0.0365409,0.49103,2.10094
5,broad_random,5,global_random,base_cfg,0.629056,0.0523773,0.619743,2.11871
6,broad_random,6,global_random,base_cfg,0.624508,0.0455817,0.54876,2.06901
7,broad_random,7,global_random,base_cfg,0.615126,0.0703738,0.499746,2.14088
8,broad_random,8,global_random,base_cfg,0.60218,0.00816438,0.524475,2.16803
9,broad_random,9,global_random,base_cfg,0.595338,0.0500284,0.488401,2.16047


## Best Discovered Candidate vs. Baseline

The search should be judged against the normal baseline, not against inherited settings from a previous workflow.
The next cell compares the single baseline run (`base_cfg`) with the best newly discovered candidate.


In [5]:
# Compare the default baseline against the best newly discovered candidate.
nrow(dense_nn_direct_search.baseline_results_df) == 1 || error("Expected exactly one baseline row.")
nrow(dense_nn_direct_search.results_df) > 0 || error("No discovered candidates available. Increase NN_DIRECT_METHOD_CANDIDATES.")

best_direct_baseline = dense_nn_direct_search.baseline_results_df[1, :]
best_direct_candidate = sort(
    copy(dense_nn_direct_search.results_df),
    [:mean_balanced_accuracy, :mean_macro_f1, :std_balanced_accuracy];
    rev = [true, true, false],
)[1, :]

best_direct_compare_df = DataFrame(
    candidate_bucket = ["baseline_base_cfg", "best_discovered_candidate"],
    search_method = [best_direct_baseline.search_method, best_direct_candidate.search_method],
    method_rank = [best_direct_baseline.method_rank, best_direct_candidate.method_rank],
    candidate_source = [best_direct_baseline.candidate_source, best_direct_candidate.candidate_source],
    seed_label = [best_direct_baseline.seed_label, best_direct_candidate.seed_label],
    mean_balanced_accuracy = [best_direct_baseline.mean_balanced_accuracy, best_direct_candidate.mean_balanced_accuracy],
    mean_macro_f1 = [best_direct_baseline.mean_macro_f1, best_direct_candidate.mean_macro_f1],
    mean_train_time_s = [best_direct_baseline.mean_train_time_s, best_direct_candidate.mean_train_time_s],
)

best_direct_candidate_metrics_df = DataFrame(
    dataset = ["real_labeled_eval", "simulated_train_set"],
    balanced_accuracy = [
        best_direct_candidate.mean_balanced_accuracy,
        best_direct_candidate.mean_sim_balanced_accuracy,
    ],
    macro_f1 = [
        best_direct_candidate.mean_macro_f1,
        best_direct_candidate.mean_sim_macro_f1,
    ],
    recall_no_class = [
        best_direct_candidate.mean_recall_no_class,
        best_direct_candidate.mean_sim_recall_no_class,
    ],
    recall_sigmoid = [
        best_direct_candidate.mean_recall_sigmoid,
        best_direct_candidate.mean_sim_recall_sigmoid,
    ],
)

display(best_direct_compare_df)
best_direct_candidate_metrics_df


Row,candidate_bucket,search_method,method_rank,candidate_source,seed_label,mean_balanced_accuracy,mean_macro_f1,mean_train_time_s
,String,String,Int64,String,String,Float64,Float64,Float64
1,baseline_base_cfg,baseline,1,baseline,base_cfg,0.673836,0.600926,13.169
2,best_discovered_candidate,latin_hypercube,1,lhs_local_base,base_cfg,0.710955,0.658715,2.12422


Row,dataset,balanced_accuracy,macro_f1,recall_no_class,recall_sigmoid
,String,Float64,Float64,Float64,Float64
1,real_labeled_eval,0.710955,0.658715,0.903392,0.518519
2,simulated_train_set,1.0,1.0,1.0,1.0


## Parameter-Level Interpretation

The next cell compares the best discovered candidate against the normal baseline.
This is the most direct way to see which simulator parameters actually moved during the direct search.


In [6]:
# Compare the best discovered candidate against the base_cfg parameter values.
best_direct_base_cfg = SmallScaleERPClassification.make_base_config(
    target_size = dense_nn_direct_search.target_size,
    apply_lowpass = dense_nn_direct_search.lowpass,
)

best_direct_param_infos = SmallScaleERPClassification.parameter_infos(best_direct_base_cfg)

best_direct_param_compare_df = DataFrame(
    parameter = [info.label for info in best_direct_param_infos],
    field = [String(info.field) for info in best_direct_param_infos],
    column = [String(info.symbol) for info in best_direct_param_infos],
    baseline_value = [Float64(best_direct_baseline[info.symbol]) for info in best_direct_param_infos],
    best_discovered_value = [Float64(best_direct_candidate[info.symbol]) for info in best_direct_param_infos],
    delta_vs_baseline = [
        Float64(best_direct_candidate[info.symbol]) - Float64(best_direct_baseline[info.symbol])
        for info in best_direct_param_infos
    ],
)

show(
    stdout,
    MIME"text/plain"(),
    best_direct_param_compare_df;
    allrows = true,
    allcols = true,
    truncate = 0,
)
println()


48×6 DataFrame
 Row │ parameter                                             field   column                               baseline_value  best_discovered_value  delta_vs_baseline 
     │ String                                                String  String                               Float64         Float64                Float64           
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ sim.mu_dist.mu                                        mean    sim_mu_mean                                  4.5                 4.63658          0.136576
   2 │ sim.mu_dist.sigma                                     std     sim_mu_std                                   0.3                 0.319337         0.0193374
   3 │ sim.sigma_dist.mu                                     mean    sim_sigma_mean                               0.5                 0.574091         0.0740913
   4 │ sim.

## Best Overall Run Summary

The final cell selects the best overall candidate across the baseline and the discovered candidates.
It then reports real-data and simulated-data metrics, bootstrap confidence intervals on the real-data metrics, and the full parameter vector.


In [7]:
# Best overall Dense-NN low-pass run summary.
all_df = dense_nn_direct_search.all_results_df
best_overall = sort(
    copy(all_df),
    [:mean_balanced_accuracy, :mean_macro_f1, :std_balanced_accuracy];
    rev = [true, true, false],
)[1, :]

println("=" ^ 80)
println("BEST DENSE-NN LOW-PASS RUN  |  Resolution: $(best_overall.resolution)  |  Low-pass: $(best_overall.lowpass)")
println("=" ^ 80)
println()
println("Candidate origin")
println("  search_method      : $(best_overall.search_method)")
println("  candidate_source   : $(best_overall.candidate_source)")
println("  seed_label         : $(best_overall.seed_label)")
println("  seed_origin        : $(best_overall.seed_origin)")
println("  source_strategy    : $(best_overall.source_strategy)")
println("  source_setting_rank: $(best_overall.source_setting_rank)")
println("  is_seed_config     : $(best_overall.is_seed_config)")
println("  eval_repeats       : $(best_overall.eval_repeats)")
println("  n_per_pattern      : $(best_overall.n_per_pattern)")
println()

perf_df = DataFrame(
    dataset = ["real_labeled_eval", "simulated_train_set"],
    balanced_accuracy = [
        best_overall.mean_balanced_accuracy,
        best_overall.mean_sim_balanced_accuracy,
    ],
    macro_f1 = [
        best_overall.mean_macro_f1,
        best_overall.mean_sim_macro_f1,
    ],
    recall_no_class = [
        best_overall.mean_recall_no_class,
        best_overall.mean_sim_recall_no_class,
    ],
    recall_sigmoid = [
        best_overall.mean_recall_sigmoid,
        best_overall.mean_sim_recall_sigmoid,
    ],
)
display(perf_df)
println()

println("-" ^ 80)
println("Bootstrap 95% CI on real-data metrics (B=2000)")
println("-" ^ 80)

best_run_base_cfg = SmallScaleERPClassification.make_base_config(
    target_size = dense_nn_direct_search.target_size,
    apply_lowpass = dense_nn_direct_search.lowpass,
)
best_cfg = SmallScaleERPClassification.build_cfg_from_settings_row(best_run_base_cfg, best_overall)
target_size = dense_nn_direct_search.target_size
key = SmallScaleERPClassification.condition_key(target_size, dense_nn_direct_search.lowpass)
real_ds = dense_nn_direct_search.real_bundle.cache[key]

sim_imgs, sim_labels = SmallScaleERPClassification.generate_single_condition_images(
    best_cfg,
    dense_nn_direct_search.n_per_pattern;
    target_size = target_size,
    lowpass = dense_nn_direct_search.lowpass,
    seed = BASE_SEED + 999_999,
)
sim_features = SmallScaleERPClassification.flatten_images(sim_imgs)

best_pred, _ = SmallScaleERPClassification.train_and_predict(
    :dense_nn_gpu,
    sim_features,
    sim_labels,
    real_ds.features;
    nn_batch_size = NN_BATCH_SIZE,
    nn_epochs = NN_EPOCHS,
    nn_lr = NN_LR,
    seed = BASE_SEED + 888_888,
)
best_y_true = Int.(real_ds.labels)
best_y_pred = Int.(best_pred)

boot_rng = Random.Xoshiro(BASE_SEED + 777_777)
B = 2000
n = length(best_y_true)
boot_ba = Vector{Float64}(undef, B)
boot_f1 = Vector{Float64}(undef, B)

for b in 1:B
    idx = rand(boot_rng, 1:n, n)
    m = SmallScaleERPClassification.evaluate_binary_metrics(best_y_true[idx], best_y_pred[idx])
    boot_ba[b] = m.balanced_accuracy
    boot_f1[b] = m.macro_f1
end

sort!(boot_ba)
sort!(boot_f1)
ci_lo = max(1, Int(floor(0.025 * B)))
ci_hi = min(B, Int(ceil(0.975 * B)))

bootstrap_df = DataFrame(
    metric = ["balanced_accuracy", "macro_f1"],
    point_estimate = [mean(boot_ba), mean(boot_f1)],
    ci_lower_2_5 = [boot_ba[ci_lo], boot_f1[ci_lo]],
    ci_upper_97_5 = [boot_ba[ci_hi], boot_f1[ci_hi]],
)
display(bootstrap_df)
println()

best_run_param_infos = SmallScaleERPClassification.parameter_infos(best_run_base_cfg)
param_df = DataFrame(
    parameter = [info.label for info in best_run_param_infos],
    value = [Float64(best_overall[info.symbol]) for info in best_run_param_infos],
    range_lo = [info.range[1] for info in best_run_param_infos],
    range_hi = [info.range[2] for info in best_run_param_infos],
)

show(stdout, MIME"text/plain"(), param_df; allrows = true, allcols = true, truncate = 0)
println()
println()
println("=" ^ 80)
println("Summary: best balanced_accuracy on real data = $(round(best_overall.mean_balanced_accuracy, digits = 4))")
println("         bootstrap 95% CI                    = [$(round(boot_ba[ci_lo], digits = 4)), $(round(boot_ba[ci_hi], digits = 4))]")
println("         best macro_f1 on real data          = $(round(best_overall.mean_macro_f1, digits = 4))")
println("         bootstrap 95% CI                    = [$(round(boot_f1[ci_lo], digits = 4)), $(round(boot_f1[ci_hi], digits = 4))]")
println("         candidate type                      = $(best_overall.is_seed_config ? "baseline" : "discovered")")
println("         image format                        = $(best_overall.resolution) pixels")
println("=" ^ 80)


Row,dataset,balanced_accuracy,macro_f1,recall_no_class,recall_sigmoid
,String,Float64,Float64,Float64,Float64
1,real_labeled_eval,0.710955,0.658715,0.903392,0.518519
2,simulated_train_set,1.0,1.0,1.0,1.0


BEST DENSE-NN LOW-PASS RUN  |  Resolution: 16x16  |  Low-pass: true

Candidate origin
  search_method      : latin_hypercube
  candidate_source   : lhs_local_base
  seed_label         : base_cfg
  seed_origin        : baseline
  source_strategy    : base_cfg
  source_setting_rank: 0
  is_seed_config     : false
  eval_repeats       : 3
  n_per_pattern      : 1000


--------------------------------------------------------------------------------
Bootstrap 95% CI on real-data metrics (B=2000)
--------------------------------------------------------------------------------
    dense_nn epoch 1/30 | loss=0.3931
    dense_nn epoch 6/30 | loss=0.01464
    dense_nn epoch 12/30 | loss=0.01039
    dense_nn epoch 18/30 | loss=0.02586
    dense_nn epoch 24/30 | loss=0.01166
    dense_nn epoch 30/30 | loss=0.01793

48×4 DataFrame
 Row │ parameter                                             value        range_lo  range_hi 
     │ String                                                Float64      Fl

Row,metric,point_estimate,ci_lower_2_5,ci_upper_97_5
,String,Float64,Float64,Float64
1,balanced_accuracy,0.785761,0.701123,0.866027
2,macro_f1,0.731853,0.661272,0.79861
